# Restricted Boltzmann Machine on Fashion-MNIST

This notebook is the step-by-step form of `rbm_fashion_mnist_libs.py`. It trains a binary Restricted Boltzmann Machine with the contrastive-divergence updates written out explicitly.

NumPy performs every weight update. TensorFlow appears in one place: `tf.keras.datasets.fashion_mnist.load_data()`, which downloads the dataset. The model, the Gibbs sampler, and the training loop do not use TensorFlow.

The walk-through follows the code in the order it runs:

1. Download Fashion-MNIST and scale pixels to probabilities.
2. Define the logistic sigmoid used by both conditionals.
3. Set the energy, the weights, and the biases.
4. Write $P(h \mid v)$ and $P(v \mid h)$.
5. Draw one block-Gibbs step.
6. Form the CD-$k$ gradients and apply them.
7. Reconstruct images and track cross-entropy while training.


## 1. Imports

`numpy` stores $W$, $a$, $b$ and evaluates the matrix products in the conditionals. `tensorflow` is imported so the next section can download Fashion-MNIST. `Path` is where the trained weights and the reconstruction picture are written.

The small PNG helper at the bottom of the next cell draws the figures in this notebook. Jupyter shows them inline. It is not part of the RBM.


In [ ]:
from pathlib import Path
import struct
import zlib

import numpy as np
import tensorflow as tf

try:
    from IPython.display import Image, display
except ImportError:
    # Outside Jupyter, keep the PNG on disk instead of displaying it.
    class Image:
        def __init__(self, data=None, **kwargs):
            self.data = data

    def display(obj):
        path = Path("output") / "notebook_preview.png"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(obj.data)
        print(f"wrote {path.resolve()}")


def grayscale_png(image: np.ndarray) -> bytes:
    """Encode a grayscale array in [0, 1] as a PNG, using only the standard library."""
    img = np.clip(np.asarray(image, dtype=np.float32) * 255.0, 0, 255).astype(np.uint8)
    height, width = img.shape
    raw = b"".join(b"\x00" + row.tobytes() for row in img)

    def chunk(tag: bytes, data: bytes) -> bytes:
        crc = zlib.crc32(tag + data) & 0xFFFFFFFF
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", crc)

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 0, 0, 0, 0)
    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", ihdr)
        + chunk(b"IDAT", zlib.compress(raw, 9))
        + chunk(b"IEND", b"")
    )


def show_gray(image: np.ndarray) -> None:
    display(Image(data=grayscale_png(image)))


## 2. Download Fashion-MNIST

Fashion-MNIST has 60,000 training images and 10,000 test images. Each image is $28 \times 28$ grayscale, so a visible layer needs

$$
n = 28 \times 28 = 784
$$

units. `load_data()` returns integer pixels in $\{0, 1, \ldots, 255\}$. The training code scales and flattens them:

$$
x_i = \frac{\text{pixel}_i}{255} \in [0, 1],
\qquad
x \in \mathbb{R}^{784}.
$$

Those values are probabilities for binary visible units. Each time a mini-batch is presented, the fit loop draws

$$
v_i \sim \mathrm{Bernoulli}(x_i).
$$

A pixel of $0.8$ therefore becomes $1$ on about 80% of presentations. Labels are not used in the energy or the updates. They only pick one example of each class when the figures are drawn.

`tf.keras.datasets.fashion_mnist.load_data` is the only TensorFlow call in the notebook.


In [ ]:
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]


def load_fashion_mnist():
    """Download Fashion-MNIST with TensorFlow and return NumPy arrays.

    Images arrive as uint8 arrays of shape (n, 28, 28). They are scaled to
    [0, 1] and flattened to (n, 784) before training.
    """
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    x_train = np.asarray(x_train, dtype=np.float32).reshape(len(x_train), -1) / 255.0
    x_test = np.asarray(x_test, dtype=np.float32).reshape(len(x_test), -1) / 255.0
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)
    return x_train, y_train, x_test, y_test


x_train, y_train, x_test, y_test = load_fashion_mnist()
print(f"train images {x_train.shape}, labels {y_train.shape}")
print(f"test images  {x_test.shape}, labels {y_test.shape}")
print(f"pixel range  [{float(x_train.min()):.3f}, {float(x_train.max()):.3f}]")

preview_index = [int(np.flatnonzero(y_train == label)[0]) for label in range(10)]
preview = np.concatenate(
    [x_train[i].reshape(28, 28) for i in preview_index],
    axis=1,
)
print("one training image from each class:")
print(", ".join(CLASS_NAMES))
show_gray(preview)


## 3. Logistic sigmoid

Both conditional distributions are elementwise logistics,

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

The implementation branches on the sign of $x$ so the exponential always sees a negative argument:

$$
\sigma(x) =
\begin{cases}
\dfrac{1}{1 + e^{-x}} & x \ge 0, \\[1.2em]
\dfrac{e^{x}}{1 + e^{x}} & x < 0.
\end{cases}
$$

The two expressions are algebraically the same. The second one avoids computing $e^{-x}$ when $x$ is a large negative number. Useful checkpoints are $\sigma(0) = 1/2$, $\sigma(x) \to 1$ as $x \to +\infty$, and $\sigma(x) \to 0$ as $x \to -\infty$.


In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable logistic sigmoid."""
    out = np.empty_like(x, dtype=np.float32)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


check = sigmoid(np.array([-4.0, -1.0, 0.0, 1.0, 4.0], dtype=np.float32))
print("sigma([-4, -1, 0, 1, 4]) =", np.array2string(check, precision=4))


## 4. Energy and parameters

Visible units $v \in \{0,1\}^n$ are the pixels. Hidden units $h \in \{0,1\}^m$ are latent features. With the defaults used later, $n = 784$ and $m = 128$.

A joint configuration has energy

$$
E(v, h) = - a^\top v - b^\top h - v^\top W h
= - \sum_i a_i v_i - \sum_j b_j h_j - \sum_{i,j} v_i W_{ij} h_j.
$$

$W_{ij}$ ties visible unit $i$ to hidden unit $j$. The Boltzmann distribution prefers low energy:

$$
P(v, h) = \frac{1}{Z} \exp\bigl(-E(v, h)\bigr),
\qquad
Z = \sum_{v,h} \exp\bigl(-E(v, h)\bigr).
$$

$Z$ sums $2^{784+128}$ states, so the training updates below never compute it.

`__init__` draws

$$
W_{ij} \sim \mathcal{N}(0, 0.01^2)
$$

and sets $b = 0$. Visible biases are then set from the data. If $p_i$ is the average of feature $i$ over the training set,

$$
a_i = \log \frac{p_i}{1 - p_i} = \sigma^{-1}(p_i).
$$

That is the bias an isolated pixel would need in order to have mean $p_i$. $p_i$ is clipped into $[10^{-3}, 1-10^{-3}]$ so the logarithm stays finite. After this initialization, $W$ and $b$ only have to learn dependence between pixels.


In [ ]:
class RestrictedBoltzmannMachine:
    """Binary-binary RBM trained with explicit CD-k updates."""

    def __init__(
        self,
        n_visible: int,
        n_hidden: int,
        learning_rate: float = 0.1,
        cd_k: int = 1,
        seed: int = 0,
    ):
        if cd_k < 1:
            raise ValueError("cd_k must be at least 1")
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        self.learning_rate = learning_rate
        self.cd_k = cd_k
        self.rng = np.random.default_rng(seed)

        # W_ij ~ Normal(0, 0.01). Visible bias is set from the data mean below.
        self.W = self.rng.normal(0.0, 0.01, size=(n_visible, n_hidden)).astype(np.float32)
        self.a = np.zeros(n_visible, dtype=np.float32)  # visible bias
        self.b = np.zeros(n_hidden, dtype=np.float32)  # hidden bias


def init_visible_bias(self, data: np.ndarray) -> None:
    """a_i = log(p_i / (1 - p_i)), with p_i the mean activation of unit i."""
    p = np.clip(data.mean(axis=0), 1e-3, 1.0 - 1e-3)
    self.a = np.log(p / (1.0 - p)).astype(np.float32)


RestrictedBoltzmannMachine.init_visible_bias = init_visible_bias

rbm = RestrictedBoltzmannMachine(
    n_visible=x_train.shape[1],
    n_hidden=128,
    learning_rate=0.1,
    cd_k=1,
    seed=0,
)
rbm.init_visible_bias(x_train)
print("W", rbm.W.shape, " a", rbm.a.shape, " b", rbm.b.shape)
print("visible bias a[:4] =", np.array2string(rbm.a[:4], precision=3))


## 5. Conditional distributions

An RBM has no edges inside the visible layer and no edges inside the hidden layer, so the conditionals factor into independent Bernoullis:

$$
P(h \mid v) = \prod_{j=1}^{m} P(h_j \mid v),
\qquad
P(v \mid h) = \prod_{i=1}^{n} P(v_i \mid h).
$$

Each factor is a sigmoid of a linear score. For one image,

$$
P(h_j = 1 \mid v) = \sigma\Bigl(b_j + \sum_{i=1}^{n} v_i W_{ij}\Bigr),
$$

$$
P(v_i = 1 \mid h) = \sigma\Bigl(a_i + \sum_{j=1}^{m} W_{ij} h_j\Bigr).
$$

For a mini-batch $V$ whose rows are images, and a hidden batch $H$, those sums are the matrix products

$$
P(h = 1 \mid v) = \sigma(V W + b),
\qquad
P(v = 1 \mid h) = \sigma(H W^\top + a).
$$

`p_h_given_v` and `p_v_given_h` are those two lines. A Bernoulli sample from a probability $p$ is the comparison

$$
u \sim \mathrm{Uniform}(0, 1),
\qquad
\text{sample} = \mathbf{1}[u < p].
$$

Binary samples are what move a Gibbs chain from one state to the next. The gradient later uses the probabilities themselves, because $\mathbb{E}[h_j \mid v] = P(h_j = 1 \mid v)$.


In [ ]:
def p_h_given_v(self, v: np.ndarray) -> np.ndarray:
    """P(h = 1 | v) = sigmoid(v W + b). v is (batch, n_visible)."""
    return sigmoid(v @ self.W + self.b)


def p_v_given_h(self, h: np.ndarray) -> np.ndarray:
    """P(v = 1 | h) = sigmoid(h W^T + a). h is (batch, n_hidden)."""
    return sigmoid(h @ self.W.T + self.a)


def sample_bernoulli(self, probs: np.ndarray) -> np.ndarray:
    """Draw each unit independently: 1 when a uniform random number is below p."""
    return (self.rng.random(probs.shape) < probs).astype(np.float32)


RestrictedBoltzmannMachine.p_h_given_v = p_h_given_v
RestrictedBoltzmannMachine.p_v_given_h = p_v_given_h
RestrictedBoltzmannMachine.sample_bernoulli = sample_bernoulli

v_example = (x_train[:1] >= 0.5).astype(np.float32)
h_example = rbm.p_h_given_v(v_example)
print("P(h=1 | v) shape", h_example.shape)
print("mean hidden probability", float(h_example.mean()))


## 6. One step of block Gibbs sampling

Gibbs sampling draws from a joint distribution by resampling each block from its conditional given the other block. On an RBM the blocks are the whole hidden layer and the whole visible layer:

$$
h^{(t)} \sim P(h \mid v^{(t)}),
\qquad
v^{(t+1)} \sim P(v \mid h^{(t)}).
$$

If $(v, h)$ is already a draw from the model joint $P(v, h)$, one more step is still a draw from that joint. Started from a training image, the chain only approaches the model distribution after many steps. Contrastive divergence, in the next section, stops the chain early on purpose.

`gibbs_hvh` starts from a hidden sample and takes the visible half of the transition, then scores the new visible state:

1. $P(v \mid h) = \sigma(h W^\top + a)$.
2. Draw $v' \sim \mathrm{Bernoulli}\bigl(P(v \mid h)\bigr)$.
3. Return $v'$ together with $P(h \mid v') = \sigma(v' W + b)$.

The returned hidden value is a probability. The chain's visible state is a sample. When $k > 1$, the caller draws a fresh binary $h$ from that probability before the next step, so every visible draw is conditioned on a binary hidden state.

A two-pixel sketch of the same step: $W = \begin{bmatrix} 0.5 \\ 0.5 \end{bmatrix}$, $a = 0$, $b = 0$, $v = (1, 0)$. Then $P(h=1 \mid v) = \sigma(0.5) \approx 0.622$. If the hidden sample is $1$, both reconstruction probabilities equal $\sigma(0.5) \approx 0.622$. One Gibbs step has already coupled the two pixels.


In [ ]:
def gibbs_hvh(self, h_sample: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """One full Gibbs step starting from a hidden sample.

    v ~ P(v | h), then return (v_sample, P(h | v)).
    """
    v_prob = self.p_v_given_h(h_sample)
    v_sample = self.sample_bernoulli(v_prob)
    h_prob = self.p_h_given_v(v_sample)
    return v_sample, h_prob


RestrictedBoltzmannMachine.gibbs_hvh = gibbs_hvh

h_sample = rbm.sample_bernoulli(h_example)
v_sample, h_prob_next = rbm.gibbs_hvh(h_sample)
print("visible sample", v_sample.shape, "fraction on", float(v_sample.mean()))
print("P(h=1 | v')   ", h_prob_next.shape, "mean", float(h_prob_next.mean()))


## 7. Contrastive divergence

Maximum likelihood for one image has the gradient

$$
\frac{\partial \log P(v)}{\partial W_{ij}}
=
\langle v_i h_j \rangle_{\mathrm{data}}
-
\langle v_i h_j \rangle_{\mathrm{model}},
$$

$$
\frac{\partial \log P(v)}{\partial a_i}
=
\langle v_i \rangle_{\mathrm{data}}
-
\langle v_i \rangle_{\mathrm{model}},
$$

$$
\frac{\partial \log P(v)}{\partial b_j}
=
\langle h_j \rangle_{\mathrm{data}}
-
\langle h_j \rangle_{\mathrm{model}}.
$$

The data average is easy. For a clamped image it is $v_i$ times $P(h_j = 1 \mid v)$, and $P(h_j = 1 \mid v)$ itself for the hidden bias. The model average is an expectation under $P(v, h)$, which requires the partition function or a fully mixed Gibbs chain.

Contrastive divergence (CD-$k$) replaces the model average by the state of a Gibbs chain that starts at the data and runs for only $k$ steps. Subscript $0$ means the data. Subscript $k$ means the chain after $k$ steps. With learning rate $\eta$,

$$
W \leftarrow W + \eta \bigl(\langle v h^\top \rangle_0 - \langle v h^\top \rangle_k\bigr),
$$

$$
a \leftarrow a + \eta \bigl(\langle v \rangle_0 - \langle v \rangle_k\bigr),
\qquad
b \leftarrow b + \eta \bigl(\langle h \rangle_0 - \langle h \rangle_k\bigr).
$$

`cd_gradients` builds those three differences for a mini-batch of $N$ rows.

**Positive phase.** $H = \sigma(VW + b) = P(h=1 \mid V)$. A binary $h$ is drawn from $H$ only to start the chain. The outer product uses $H$, not that sample.

**Negative phase.** Repeat $k$ times: $(v^-, H^-) = \mathrm{gibbs}(h)$. On every step except the last, replace $h$ by a fresh Bernoulli draw from $H^-$. For the default $k = 1$ this is a single step, $v \to h \to v^- \to P(h \mid v^-)$.

**Mini-batch means.**

$$
\Delta W = \frac{1}{N}\bigl(V^\top H - (v^-)^\top H^-\bigr),
\quad
\Delta a = \mathrm{mean}(V - v^-),
\quad
\Delta b = \mathrm{mean}(H - H^-).
$$

`update` applies $\eta = 0.1$ with no momentum and no weight decay:

$$
W \leftarrow W + \eta\,\Delta W,
\quad
a \leftarrow a + \eta\,\Delta a,
\quad
b \leftarrow b + \eta\,\Delta b.
$$

CD-$k$ is biased relative to the true likelihood gradient, because $k$ steps have not reached the stationary distribution. Larger `cd_k` shortens that gap and costs one extra pair of matrix products per step.


In [ ]:
def cd_gradients(self, v0: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Contrastive-divergence gradients for one mini-batch.

    Positive phase: hidden probabilities given the data.
    Negative phase: cd_k Gibbs steps. The gradient uses P(h | v_k).
    """
    # Positive phase: h_j = P(h_j = 1 | v)
    h0_prob = self.p_h_given_v(v0)
    h_sample = self.sample_bernoulli(h0_prob)

    # Negative phase. Intermediate steps draw a binary h.
    # The last hidden state used in the gradient stays a probability.
    v_neg = v0
    h_neg_prob = h0_prob
    for step in range(self.cd_k):
        v_neg, h_neg_prob = self.gibbs_hvh(h_sample)
        if step < self.cd_k - 1:
            h_sample = self.sample_bernoulli(h_neg_prob)

    batch = v0.shape[0]
    # <v h^T>_data - <v h^T>_model
    dW = (v0.T @ h0_prob - v_neg.T @ h_neg_prob) / batch
    # <v>_data - <v>_model
    da = (v0 - v_neg).mean(axis=0)
    # <h>_data - <h>_model
    db = (h0_prob - h_neg_prob).mean(axis=0)
    return dW, da, db


def update(self, v0: np.ndarray) -> None:
    """Apply W += lr * dW, a += lr * da, b += lr * db."""
    dW, da, db = self.cd_gradients(v0)
    lr = self.learning_rate
    self.W += lr * dW
    self.a += lr * da
    self.b += lr * db


RestrictedBoltzmannMachine.cd_gradients = cd_gradients
RestrictedBoltzmannMachine.update = update

# One CD-1 gradient on 128 binarized images. Weights are left unchanged.
v_batch = rbm.sample_bernoulli(x_train[:128])
dW, da, db = rbm.cd_gradients(v_batch)
print("dW", dW.shape, " mean", float(dW.mean()))
print("da", da.shape, " mean", float(da.mean()))
print("db", db.shape, " mean", float(db.mean()))
print("learning-rate step uses  W <- W + 0.1 * dW,  and the same for a and b")


## 8. Reconstruction and the monitored loss

The picture written at the end is a mean-field reconstruction, not a Gibbs sample. Each conditional draw is replaced by its probability:

$$
\hat h = P(h = 1 \mid v),
\qquad
\hat v = P(v = 1 \mid \hat h) = \sigma(\hat h\, W^\top + a).
$$

$\hat v$ is a grayscale image in $(0, 1)^{784}$. The monitor is the mean Bernoulli cross-entropy between a binary target $v$ and this reconstruction, with probabilities clipped to $[10^{-7}, 1-10^{-7}]$:

$$
\mathrm{CE}
=
-\frac{1}{Nn}\sum_{s,i}
\Bigl(
v_{si}\log \hat v_{si}
+
(1 - v_{si})\log(1 - \hat v_{si})
\Bigr).
$$

A falling CE means one pass of $v \to P(h \mid v) \to P(v \mid h)$ is assigning higher probability to the thresholded pixels. It is a reconstruction diagnostic. It is not $\log P(v)$, because that still depends on the partition function.

`fit` repeats the following for each epoch:

1. Shuffle the training rows.
2. For each block of 128 rows, draw $v_i \sim \mathrm{Bernoulli}(x_i)$ and call `update`.
3. Score CE on the first 2,048 training images, thresholded at $1/2$, so every epoch is scored on the same binary targets.


In [ ]:
def reconstruct(self, v: np.ndarray) -> np.ndarray:
    """Mean-field reconstruction: v -> P(h | v) -> P(v | h)."""
    return self.p_v_given_h(self.p_h_given_v(v))


def reconstruction_cross_entropy(self, v: np.ndarray) -> float:
    recon = np.clip(self.reconstruct(v), 1e-7, 1.0 - 1e-7)
    ce = -(v * np.log(recon) + (1.0 - v) * np.log(1.0 - recon))
    return float(ce.mean())


def fit(
    self,
    data: np.ndarray,
    epochs: int = 5,
    batch_size: int = 128,
    binarize: bool = True,
) -> list[float]:
    """Train on rows of data with values in [0, 1].

    If binarize is true, each presentation draws v_i ~ Bernoulli(pixel_i).
    """
    n = data.shape[0]
    history = []
    for epoch in range(1, epochs + 1):
        order = self.rng.permutation(n)
        for start in range(0, n, batch_size):
            batch = data[order[start : start + batch_size]]
            if binarize:
                batch = self.sample_bernoulli(batch)
            self.update(batch)

        # Fixed subset, fixed threshold, so epochs can be compared.
        probe = data[:2048]
        if binarize:
            probe = (probe >= 0.5).astype(np.float32)
        loss = self.reconstruction_cross_entropy(probe)
        history.append(loss)
        print(f"epoch {epoch:3d}/{epochs}  reconstruction cross-entropy {loss:.4f}")
    return history


RestrictedBoltzmannMachine.reconstruct = reconstruct
RestrictedBoltzmannMachine.reconstruction_cross_entropy = reconstruction_cross_entropy
RestrictedBoltzmannMachine.fit = fit


## 9. Train

The cell below builds a new machine with seed 0, sets $a$ from the training mean, and runs CD-1 for 5 epochs. The gradient demonstration above used the first machine and did not write into these parameters.

| Symbol | Value | Role |
| --- | --- | --- |
| $m$ | 128 | hidden units |
| $\eta$ | 0.1 | learning rate |
| $k$ | 1 | Gibbs steps per update |
| $N$ | 128 | mini-batch size |
| epochs | 5 | passes over the 60,000 training images |

On this machine a full run takes well under a minute after the dataset is cached.


In [ ]:
rbm = RestrictedBoltzmannMachine(
    n_visible=x_train.shape[1],
    n_hidden=128,
    learning_rate=0.1,
    cd_k=1,
    seed=0,
)
rbm.init_visible_bias(x_train)
history = rbm.fit(x_train, epochs=5, batch_size=128, binarize=True)


## 10. Reconstructions

One test image is taken from each class and thresholded at $1/2$, which is the same binarization as the monitor. `reconstruct` maps that binary image through $P(h \mid v)$ and $P(v \mid h)$.

The figure has the ten originals on the top row and the ten reconstructions on the bottom row, in the class order printed below. The same array is written to `output/reconstructions.pgm`. Weights, biases, and the CE history are written to `output/rbm_weights.npz`.

Test CE uses the first 2,048 test images, again thresholded at $1/2$.


In [ ]:
def save_pgm(path: Path, image: np.ndarray) -> None:
    """Write one grayscale image in [0, 1] as a binary PGM file."""
    img = np.clip(image * 255.0, 0, 255).astype(np.uint8)
    height, width = img.shape
    header = f"P5\n{width} {height}\n255\n".encode("ascii")
    path.write_bytes(header + img.tobytes())


def save_comparison_grid(
    path: Path,
    originals: np.ndarray,
    reconstructions: np.ndarray,
    image_shape: tuple[int, int] = (28, 28),
) -> None:
    """Stack original (top) and reconstruction (bottom) into one PGM."""
    n = originals.shape[0]
    h, w = image_shape
    gap = 2
    canvas = np.ones((h * 2 + gap, n * w), dtype=np.float32)
    for i in range(n):
        canvas[0:h, i * w : (i + 1) * w] = originals[i].reshape(h, w)
        canvas[h + gap :, i * w : (i + 1) * w] = reconstructions[i].reshape(h, w)
    save_pgm(path, canvas)
    return canvas


picks = [int(np.flatnonzero(y_test == label)[0]) for label in range(10)]
originals = (x_test[picks] >= 0.5).astype(np.float32)
reconstructions = rbm.reconstruct(originals)

out_dir = Path("output")
out_dir.mkdir(parents=True, exist_ok=True)
grid_path = out_dir / "reconstructions.pgm"
canvas = save_comparison_grid(grid_path, originals, reconstructions)
np.savez(
    out_dir / "rbm_weights.npz",
    W=rbm.W,
    a=rbm.a,
    b=rbm.b,
    history=np.asarray(history, dtype=np.float32),
)

test_probe = (x_test[:2048] >= 0.5).astype(np.float32)
test_ce = rbm.reconstruction_cross_entropy(test_probe)
print(f"test reconstruction cross-entropy {test_ce:.4f}")
print("top row is the data, bottom row is P(v | P(h | v))")
print("class order:", ", ".join(CLASS_NAMES))
print(f"wrote {grid_path.resolve()}")
print(f"wrote {(out_dir / 'rbm_weights.npz').resolve()}")
show_gray(canvas)
